In [86]:
from __future__ import annotations

In [87]:
class Money:
    rate_to_BYN = 1
    rate_to_EUR = 1
    rate_to_USD = 1

    def __init__(self, rubles: int = 0, kopecks: int = 0, currency: str = "У. E"):
        self.rubles = rubles
        self.kopecks = kopecks
        self.currency = currency
        self._total = rubles + kopecks / 100

    @property
    def total(self):
        return self.rubles + round(self.kopecks / 100, 2)

    @total.setter
    def total(self, value: float):
        self._total = round(value, 2)
        self.rubles = int(value)
        self.kopecks = int((value - self.rubles) * 100)

    def to_string(self) -> str:
        return f"{self.rubles},{self.kopecks:02d} {self.currency}"

    def add(self, other: Money, sum: float) -> None:
        if not isinstance(other, Money):
            return
        another_sum = sum
        match other.currency:
            case "BYN":
                another_sum *= self.__class__.rate_to_BYN
            case "EUR":
                another_sum *= self.__class__.rate_to_EUR
            case "USD":
                another_sum *= self.__class__.rate_to_USD
        self.total += sum
        self.rubles = self.total // 1
        self.kopecks = int((self.total % 1) * 100)
        other.total -= another_sum
        other.rubles = other.total // 1
        other.kopecks = int((other.total % 1) * 100)

    def sub(self, other: Money, sum: float) -> None:
        if not isinstance(other, Money):
            return
        another_sum = sum
        match other.currency:
            case "BYN":
                another_sum *= self.__class__.rate_to_BYN
            case "EUR":
                another_sum *= self.__class__.rate_to_EUR
            case "USD":
                another_sum *= self.__class__.rate_to_USD
        self.total -= sum
        self.rubles = self.total // 1
        self.kopecks = int((self.total % 1) * 100)
        other.total += another_sum
        other.rubles = other.total // 1
        other.kopecks = int((other.total % 1) * 100)

    def mul(self, n: float) -> None:
        self.total = self.total * n

    def div(self, n: float) -> None:
        if n == 0:
            print("Error")
            return
        self.total = self.total / n

    def equal(self, other: Money) -> bool:
        if not isinstance(other, Money):
            return False
        if self.currency == other.currency and abs(self.total - other.total) < 0.001:
            return True
        else:
            return False

    def total_to_BYN(self) -> float:
        return round(self.total * self.__class__.rate_to_BYN, 2)

    def total_to_USD(self) -> float:
        return round(self.total * self.__class__.rate_to_USD, 2)

    def total_to_EUR(self) -> float:
        return round(self.total * self.__class__.rate_to_EUR, 2)

In [88]:
class BYN(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="BYN")
        self.__class__.rate_to_EUR = 0.3
        self.__class__.rate_to_USD = 0.36
        self.__class__.rate_to_BYN = 1

In [89]:
class USD(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="USD")
        self.__class__.rate_to_EUR = 0.85
        self.__class__.rate_to_BYN = 2.8
        self.__class__.rate_to_USD = 1

In [90]:
class EUR(Money):
    def __init__(self, rubles=0, kopecks=0):
        super().__init__(rubles, kopecks, currency="EUR")
        self.__class__.rate_to_EUR = 1
        self.__class__.rate_to_USD = 1.17
        self.__class__.rate_to_BYN = 3.28

In [91]:
class BankAccaunt:
    def __init__(self, username: str, invoices: list = None):
        self.username = username
        self.invoices = invoices if invoices is not None else []

    def add_invoice(self, currency: str, rubles: int = 0, kopecks: int = 0) -> None:
        if currency == "BYN":
            new_invoice = BYN(rubles, kopecks)
        elif currency == "USD":
            new_invoice = USD(rubles, kopecks)
        elif currency == "EUR":
            new_invoice = EUR(rubles, kopecks)
        else:
            print(f"Ошибка: неизвестная валюта '{currency}'")
            return

        self.invoices.append(new_invoice)
        print(
            f"{self.username} : Счёт {currency} на сумму {rubles},{kopecks:02d} создан и добавлен под номером {len(self.invoices)}"
        )

    def total_report(self) -> None:
        total_BYN = sum(item.total_to_BYN() for item in self.invoices)
        total_USd = sum(item.total_to_USD() for item in self.invoices)
        total_EUR = sum(item.total_to_EUR() for item in self.invoices)
        print(
            f"Общая сумма всех счетов {self.username} : {total_BYN} byn или {total_EUR} eur или {total_USd} usd"
        )

    def report_invoice(self, ind: int):
        if ind > len(self.invoices) or ind <= 0:
            print("Некорректный номер счета")
            return
        print(
            f"На {ind} счете {self.username} лежит: {self.invoices[ind-1].total_to_BYN()} byn или {self.invoices[ind-1].total_to_EUR()} eur или {self.invoices[ind-1].total_to_USD()} usd"
        )

    def send(self, from_ind: int, value: float, to_invoice: Money):
        if from_ind >= len(self.invoices) or from_ind < 0:
            print("Некорректный номер счета")
            return

        self.invoices[from_ind - 1].sub(to_invoice, value)
        print(
            f"Co cчёта {from_ind} {self.username} совершил перевод на сумму {value} {self.invoices[from_ind-1].currency} на счет {to_invoice.currency}"
        )

In [92]:
saveliys_accaunt = BankAccaunt("Saveliy")
andreys_accaunt = BankAccaunt("Andrew")
saveliys_accaunt.add_invoice("BYN", 5, 50)
saveliys_accaunt.add_invoice("USD", 5, 50)
andreys_accaunt.add_invoice("EUR", 5, 50)
saveliys_accaunt.report_invoice(1)
saveliys_accaunt.total_report()
saveliys_accaunt.send(1, 5, andreys_accaunt.invoices[0])
saveliys_accaunt.total_report()
andreys_accaunt.total_report()

Saveliy : Счёт BYN на сумму 5,50 создан и добавлен под номером 1
Saveliy : Счёт USD на сумму 5,50 создан и добавлен под номером 2
Andrew : Счёт EUR на сумму 5,50 создан и добавлен под номером 1
На 1 счете Saveliy лежит: 5.5 byn или 1.65 eur или 1.98 usd
Общая сумма всех счетов Saveliy : 20.9 byn или 6.32 eur или 7.48 usd
Co cчёта 1 Saveliy совершил перевод на сумму 5 BYN на счет EUR
Общая сумма всех счетов Saveliy : 15.9 byn или 4.82 eur или 5.68 usd
Общая сумма всех счетов Andrew : 22.96 byn или 7.0 eur или 8.19 usd
